# 05 · Full scale on a GPU (Colab / Kaggle)

The same code, at a scale this 4-core laptop cannot reach, plus the optional real-data path.

**This notebook is not executed in CI and its outputs are not committed** — it needs a GPU runtime and, for the real-data section, a dataset the repository deliberately does not download. Everything shipped in `results/` comes from the CPU path in notebooks 01-04.

## Setup

Uncomment on a fresh Colab runtime. The dependency set is deliberately small: torch, numpy, pandas, pyyaml, matplotlib. No pyannote, no speechbrain, no scipy.

In [ ]:
# !git clone https://github.com/arslan-ahmad/streaming-speaker-diarization.git
# %cd streaming-speaker-diarization
# !pip install -q torch numpy pandas pyyaml matplotlib tqdm
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
import torch
print('cuda available:', torch.cuda.is_available())

## Scale up

The shipped config is sized for a CPU budget: 900 training steps, 24 test recordings of 60 s, a 40-channel front end. On a GPU all of those can grow by an order of magnitude with no code change — every knob is a config field, and `--set` overrides are typed by the target field so a bool stays a bool.

In [ ]:
from streamdiar.config import load_config
cfg = load_config('configs/base.yaml', [
    'train.steps=9000',
    'train.n_speakers_per_batch=48',
    'train.threads=8',
    'embedder.channels=192',
    'embedder.dilations=1,2,4,8,16,32',
    'embedder.embed_dim=128',
    'data.n_features=64',
    'data.duration_s=300',
    'data.n_train_speakers=1200',
    'data.n_test_recordings=100',
])
print('receptive field will be',
      1 + 2 * sum(cfg.embedder.dilations), 'frames')
print('test audio:',
      cfg.data.n_test_recordings * cfg.data.duration_s / 3600, 'hours')

In [ ]:
# from streamdiar.engine import train_embedder, fit_vad
# from streamdiar.data.generator import generate_split
# model = train_embedder(cfg, seed=0, progress=True)
# fit_vad(model, cfg, generate_split(cfg.data, 0, 'dev'))

## The real-data path

`streamdiar.data.real` turns a WAV plus its RTTM reference into the same `Recording` object the generator produces, through a hand-rolled log-mel front end (25 ms Hann frames, 10 ms hop, 40 triangular mel filters, **natural** log so the units match the generator exactly). Every diarizer and every metric then works unchanged.

The caveat that must travel with any number produced this way: an RTTM reference is a human annotation with boundary error of tens of milliseconds, which is the same order as the differences the latency sweep resolves. That is precisely why the shipped results use generated data with an exact reference.

In [ ]:
# from streamdiar.data.real import load_real_split
# from streamdiar.engine import evaluate, aggregate
# recs = load_real_split('data/ami', frame_rate=100, n_mels=cfg.data.n_features)
# for method in ['online', 'offline_ahc', 'offline_spectral']:
#     agg = aggregate(evaluate(model, cfg, recs, method=method))
#     print(f"{method:18s} DER {agg['der']:.4f}  "
#           f"latency {agg['latency_median_ms']:.0f} ms")
print('real-data cells are commented out: no dataset is bundled')

## The latency sweep at scale

The interesting question a GPU run can answer that this laptop cannot: does the optimum of the DER-versus-latency curve move when the embedder is strong enough that single-window embeddings are already reliable? The mechanism's benefit is variance reduction on the query, so a lower-variance embedder should need *less* lookahead — the optimum should move left. That is a prediction, not a result: it has not been run here, and it is listed as open work in `docs/RESULTS.md`.

In [ ]:
# from streamdiar.pipelines.latency import run_latency_sweep, summarise_curve
# tbl = run_latency_sweep(cfg, seeds=(0, 1, 2))
# print(summarise_curve(tbl).to_string(index=False))